# 04 · Baseline: ResNet-50 — regresión de visibilidad

**Camanchaca-Predict — G5 · Proyecto Aplicado 2026-2**

Entrenamos el **baseline exigido por la rúbrica**: ResNet-50 preentrenada en
ImageNet con cabeza de regresión sobre `log10(V)` estandarizado.

**Diseño del baseline:**

| Decisión | Justificación |
|---|---|
| ResNet-50 (ImageNet-1k) | Arquitectura conocida, rápida en T4, comparación justa |
| Target `y_norm` = (log10 V − μ)/σ | Estabiliza el rango 10–2000 m; error MSE ≈ error relativo |
| Cabeza MLP (2048→256→1) | Capacidad suficiente para regresión fina |
| Aumentación SOLO geométrica | Jitter de color/brillo alteraría la apariencia de la niebla |
| Bandas derivadas del V̂ predicho | F1 macro sin entrenar un clasificador separado |

Produce: `checkpoints/resnet50_best.pt`, `results/resnet50_metrics.json` y
figuras para la presentación.

Requiere: splits del notebook 03. GPU T4 recomendada (~25 min).

In [ ]:
# Setup estándar del proyecto
import json, math, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/camanchaca")
except ImportError:
    ROOT = Path("local_workspace")

DATA_DIR = ROOT / "data"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
FIG_DIR = ROOT / "figures"
for p in (CKPT_DIR, RESULTS_DIR, FIG_DIR):
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
print(f"Dispositivo: {DEVICE}")

In [ ]:
# Cargamos splits y estadísticas del target (notebook 03)
train_df = pd.read_csv(DATA_DIR / "splits" / "train.csv")
val_df = pd.read_csv(DATA_DIR / "splits" / "val.csv")
test_df = pd.read_csv(DATA_DIR / "splits" / "test.csv")
with open(DATA_DIR / "splits" / "class_weights.json") as f:
    stats = json.load(f)
MU, SIGMA = stats["mu"], stats["sigma"]
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)} · MU={MU:.3f} SIGMA={SIGMA:.3f}")

In [ ]:
# Dataset y transformaciones (sin jitter de color: alteraría la niebla)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.Resize(256), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class VisibilityDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = self.transform(Image.open(r["path"]).convert("RGB"))
        return (img, float(r["y_norm"]), int(r["band_idx"]), float(r["visibility_m"]))

BATCH = 64
train_loader = DataLoader(VisibilityDataset(train_df, train_tf), batch_size=BATCH,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(VisibilityDataset(val_df, eval_tf), batch_size=BATCH,
                        num_workers=2, pin_memory=True)
test_loader = DataLoader(VisibilityDataset(test_df, eval_tf), batch_size=BATCH,
                         num_workers=2, pin_memory=True)
print(f"Loaders listos · batch={BATCH}")

In [ ]:
# Modelo: ResNet-50 + cabeza de regresión (idéntico a src/camanchaca/models/)
class ResNetVisibility(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        net = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2
                              if pretrained else None)
        net.fc = nn.Identity()
        self.backbone = net
        self.head = nn.Sequential(nn.Linear(2048, 256), nn.ReLU(inplace=True),
                                  nn.Dropout(0.2), nn.Linear(256, 1))
    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(-1)

model = ResNetVisibility(pretrained=True).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"ResNet-50 · {n_params/1e6:.1f} M parámetros")

In [ ]:
# Métricas y funciones de época (log10 V -> metros)
BAND_EDGES = (50.0, 100.0, 200.0)
BAND_NAMES = ("critico", "alto_riesgo", "precaucion", "aceptable")

def to_meters(y_norm):
    return 10 ** (y_norm * SIGMA + MU)

def compute_metrics(v_true, v_pred):
    from sklearn.metrics import f1_score
    err = v_pred - v_true
    mae = float(np.abs(err).mean())
    rmse = float(np.sqrt((err ** 2).mean()))
    mape = float((np.abs(err) / np.clip(v_true, 1e-6, None)).mean() * 100)
    ss_res = float((err ** 2).sum())
    ss_tot = float(((v_true - v_true.mean()) ** 2).sum())
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    b_true = np.digitize(v_true, BAND_EDGES)
    b_pred = np.digitize(v_pred, BAND_EDGES)
    f1 = float(f1_score(b_true, b_pred, average="macro", zero_division=0))
    peligro = v_true < 100.0
    falso_seguro = float((v_pred[peligro] >= 100.0).mean()) if peligro.any() else 0.0
    return {"mae_m": mae, "rmse_m": rmse, "mape_pct": mape, "r2": r2,
            "f1_macro": f1, "false_safe_rate": falso_seguro}

def run_epoch(model, loader, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    losses, preds, vtrues = [], [], []
    with torch.set_grad_enabled(training):
        for x, y_norm, band, v_true in loader:
            x, y_norm = x.to(DEVICE, non_blocking=True), y_norm.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16,
                                enabled=training and scaler is not None):
                out = model(x)
                loss = nn.functional.mse_loss(out, y_norm)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            losses.append(loss.item() * x.size(0))
            preds.append(out.float().cpu().numpy())
            vtrues.append(v_true.numpy())
    return (np.sum(losses) / len(loader.dataset),
            to_meters(np.concatenate(preds)),
            np.concatenate(vtrues))

In [ ]:
# Entrenamiento completo con early stopping y checkpoint por época
EPOCHS = 8
PATIENCE = 3

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")

history, best_mae, best_epoch, no_improve = [], float("inf"), -1, 0
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    tr_loss, _, _ = run_epoch(model, train_loader, optimizer, scaler)
    with torch.no_grad():
        va_loss, va_pred, va_true = run_epoch(model, val_loader)
    va_m = compute_metrics(va_true, va_pred)
    history.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss, **va_m})
    marker = ""
    if va_m["mae_m"] < best_mae:
        best_mae, best_epoch, no_improve = va_m["mae_m"], epoch, 0
        torch.save({"state_dict": model.state_dict(), "mu": MU, "sigma": SIGMA,
                    "epoch": epoch, "val_metrics": va_m},
                   CKPT_DIR / "resnet50_best.pt")
        marker = "  ← mejor (guardado)"
    else:
        no_improve += 1
    print(f"[{epoch:02d}] train={tr_loss:.4f} val={va_loss:.4f} "
          f"MAE={va_m['mae_m']:6.1f} m  RMSE={va_m['rmse_m']:6.1f}  "
          f"F1={va_m['f1_macro']:.3f}  FS={va_m['false_safe_rate']:.3f}{marker}")
    scheduler.step()
    if no_improve >= PATIENCE:
        print(f"Early stopping en época {epoch} (mejor: {best_epoch})")
        break
print(f"\\nEntrenamiento: {(time.time()-t0)/60:.1f} min · mejor época {best_epoch} "
      f"(val MAE {best_mae:.1f} m)")

In [ ]:
# Curvas de aprendizaje
hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(hist["epoch"], hist["train_loss"], "o-", label="train")
axes[0].plot(hist["epoch"], hist["val_loss"], "s-", label="val")
axes[0].set_xlabel("época"); axes[0].set_ylabel("MSE (y_norm)"); axes[0].legend()
axes[0].set_title("Pérdida")
axes[1].plot(hist["epoch"], hist["mae_m"], "o-", color="#b2182b")
axes[1].set_xlabel("época"); axes[1].set_ylabel("MAE (m)")
axes[1].set_title("MAE en validación")
fig.suptitle("ResNet-50 — curvas de aprendizaje")
plt.savefig(FIG_DIR / "resnet50_curvas.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Evaluación FINAL en test con el mejor checkpoint
ckpt = torch.load(CKPT_DIR / "resnet50_best.pt", map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
with torch.no_grad():
    _, te_pred, te_true = run_epoch(model, test_loader)
resnet_metrics = compute_metrics(te_true, te_pred)
resnet_metrics.update({"model": "resnet50_baseline", "n_params_M": round(n_params/1e6, 1),
                       "best_epoch": best_epoch,
                       "train_time_min": round((time.time()-t0)/60, 1)})
print(json.dumps(resnet_metrics, indent=2))

In [ ]:
# Matriz de confusión (bandas derivadas de V̂) + F1 por banda
from sklearn.metrics import classification_report, confusion_matrix

b_true = np.digitize(te_true, BAND_EDGES)
b_pred = np.digitize(te_pred, BAND_EDGES)
cm = confusion_matrix(b_true, b_pred, labels=[0, 1, 2, 3])
fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(BAND_NAMES, rotation=25, ha="right")
ax.set_yticklabels(BAND_NAMES)
ax.set_xlabel("Predicha"); ax.set_ylabel("Real"); ax.set_title("ResNet-50 · test")
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.savefig(FIG_DIR / "resnet50_confusion.png", dpi=150, bbox_inches="tight")
plt.show()
print(classification_report(b_true, b_pred, target_names=BAND_NAMES, zero_division=0))

In [ ]:
# Exportamos métricas (Drive + repo local) para el notebook 06 y los slides
with open(RESULTS_DIR / "resnet50_metrics.json", "w") as f:
    json.dump(resnet_metrics, f, indent=2)

REPO_LOCAL = Path("/content/camanchaca-predict")
if REPO_LOCAL.exists():
    import shutil
    (REPO_LOCAL / "reports" / "figures").mkdir(parents=True, exist_ok=True)
    for fig in ("resnet50_curvas.png", "resnet50_confusion.png"):
        shutil.copy(FIG_DIR / fig, REPO_LOCAL / "reports" / "figures" / fig)
    (REPO_LOCAL / "reports" / "tables").mkdir(parents=True, exist_ok=True)
    shutil.copy(RESULTS_DIR / "resnet50_metrics.json",
                REPO_LOCAL / "reports" / "tables" / "resnet50_metrics.json")
print("Métricas y figuras exportadas ✅")

## Checklist de interpretación (para la presentación)

- ¿La curva de val sigue a la de train? (sobreajuste → R8)
- ¿El MAE en metros es aceptable para la banda donde más falla?
- **Falso-seguro**: ¿cuántos casos con V real < 100 m se predijeron ≥ 100 m?
  Es el número más importante para seguridad vial.
- La matriz de confusión muestra ¿dónde se concentran los errores? (típico:
  bandas adyacentes, que es el error "menos grave").

**Siguiente:** notebook 05 · vit_finetune — el modelo protagonista con
cabeza dual (regresión + bandas).